# EDA — Customer Support Ticket Dataset

**Objetivo:** realizar a análise exploratória inicial do dataset de atendimento ao cliente e definir uma base para futura classificação automática de intenções.

**Fonte:** Kaggle — Customer Support Ticket Dataset, de Suraj. O dataset possui 8.469 registros e 17 atributos. A documentação oficial informa que os dados cobrem tickets sobre problemas técnicos, cobrança, produto, acesso, perda de dados e outros temas de suporte.

**Motivo da escolha:** o dataset conecta texto de atendimento (`Ticket Subject`/`Ticket Description`) com uma categoria de ticket (`Ticket Type`), permitindo futuramente transformar a análise em um problema de classificação de intenção.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("../data/customer_support_tickets.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    import kagglehub
    path = kagglehub.dataset_download("suraj520/customer-support-ticket-dataset")
    csv_path = next(Path(path).glob("*.csv"))
    df = pd.read_csv(csv_path)

print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

## 1. Compreensão do problema e do dataset

Cada linha representa um ticket de atendimento. As variáveis incluem dados demográficos, produto adquirido, data de compra, tipo e assunto do ticket, descrição, status, prioridade, canal, tempos de atendimento e satisfação.

A futura API terá uma rota `/predict` que receberá texto e devolverá uma intenção. Nesta etapa, a intenção é simulada; o objetivo é entender a distribuição dos dados antes de construir um modelo de ML/NLP.

In [ ]:
df.info()

In [ ]:
print("Colunas:")
for col in df.columns:
    print(f"- {col}")

In [ ]:
df.describe(include="all").T

## 2. Inspeção inicial

O dataset original tem 8.469 tickets e 17 colunas. A documentação publicada no Kaggle identifica `Ticket ID` como identificador único e descreve as demais variáveis como atributos do cliente, produto, atendimento e satisfação.

In [ ]:
print("Duplicatas de linhas:", df.duplicated().sum())
print("IDs duplicados:", df["Ticket ID"].duplicated().sum())
print("Valores únicos por coluna:")
df.nunique().sort_values()

## 3. Qualidade dos dados

É esperado que algumas variáveis tenham valores ausentes de forma estrutural: tickets não encerrados podem não possuir resolução, tempo de resolução ou avaliação de satisfação; tickets sem resposta ainda podem não possuir tempo de primeira resposta. Portanto, ausência nesses campos não deve ser tratada automaticamente como erro.

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
missing[missing["missing_count"] > 0]

In [ ]:
# Validações de domínio
checks = {
    "idade_fora_de_18_a_70": ((df["Customer Age"] < 18) | (df["Customer Age"] > 70)).sum(),
    "satisfacao_fora_de_1_a_5": (~df["Customer Satisfaction Rating"].dropna().between(1, 5)).sum(),
    "ticket_id_duplicado": df["Ticket ID"].duplicated().sum(),
}
checks

### Observação importante sobre o texto

Há descrições com placeholders como `{product_purchased}` e `{error_message}`. Isso é uma limitação de qualidade relevante para uma futura tarefa de NLP: o texto pode conter templates repetidos e reduzir a informação lexical disponível. A limpeza pode preservar o texto original para auditoria e, em uma etapa futura, criar uma versão normalizada para modelagem.

## 4. Limpeza e preparação

A preparação abaixo é conservadora: não inventa valores ausentes. Datas são convertidas para `datetime`, espaços são normalizados em colunas textuais e uma cópia é criada para análise. Campos cuja ausência é semântica são mantidos como `NaN`.

In [ ]:
eda = df.copy()

# Padronização de texto
text_cols = eda.select_dtypes(include="object").columns
for col in text_cols:
    eda[col] = eda[col].astype("string").str.strip()

# Datas
for col in ["Date of Purchase", "First Response Time", "Time to Resolution"]:
    eda[col] = pd.to_datetime(eda[col], errors="coerce")

# Variáveis derivadas úteis
eda["Ticket Age Days"] = (eda["Date of Purchase"].max() - eda["Date of Purchase"]).dt.days
eda["Description Length"] = eda["Ticket Description"].fillna("").str.len()

print(eda.dtypes)

In [ ]:
# Checagem pós-preparação
print("Duplicatas:", eda.duplicated().sum())
print("IDs duplicados:", eda["Ticket ID"].duplicated().sum())
print("Datas inválidas/nulas após conversão:")
print(eda[["Date of Purchase", "First Response Time", "Time to Resolution"]].isna().sum())

## 5. Análise Univariada

A análise concentra-se nas variáveis que melhor descrevem o comportamento dos tickets: idade, satisfação, tipo, assunto, status, prioridade e canal. Para variáveis categóricas, barras mostram frequência; para variáveis numéricas, histogramas mostram distribuição.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(eda["Customer Age"], bins=15, kde=True, ax=ax)
ax.set_title("Distribuição da idade dos clientes")
ax.set_xlabel("Idade")
ax.set_ylabel("Quantidade de tickets")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(eda["Customer Satisfaction Rating"].dropna(), discrete=True, ax=ax)
ax.set_title("Distribuição da satisfação")
ax.set_xlabel("Nota de satisfação")
ax.set_ylabel("Quantidade de tickets encerrados avaliados")
plt.tight_layout()
plt.show()

In [ ]:
def plot_count(column, title, top_n=None, rotate=0):
    counts = eda[column].value_counts()
    if top_n:
        counts = counts.head(top_n)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=counts.values, y=counts.index, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Quantidade")
    ax.set_ylabel(column)
    plt.tight_layout()
    plt.show()
    return counts

plot_count("Ticket Type", "Distribuição por tipo de ticket")
plot_count("Ticket Priority", "Distribuição por prioridade")
plot_count("Ticket Channel", "Distribuição por canal")
plot_count("Ticket Status", "Distribuição por status")
plot_count("Ticket Subject", "Principais assuntos dos tickets", top_n=10)

In [ ]:
# Frequências numéricas/categóricas em tabela para documentação
print("Ticket Type\n", eda["Ticket Type"].value_counts(), "\n")
print("Ticket Priority\n", eda["Ticket Priority"].value_counts(), "\n")
print("Ticket Channel\n", eda["Ticket Channel"].value_counts(), "\n")
print("Ticket Status\n", eda["Ticket Status"].value_counts(), "\n")
print("Ticket Subject — top 10\n", eda["Ticket Subject"].value_counts().head(10))

## Hipóteses sobre as intenções dos usuários

**H1 — As intenções de suporte se concentram em poucos temas recorrentes.** A presença de assuntos como `Refund request`, `Software bug`, `Product compatibility`, `Delivery problem`, `Hardware issue`, `Battery life`, `Network problem`, `Installation support`, `Product setup` e `Payment issue` sugere que a maior parte dos tickets pode ser organizada em um conjunto relativamente pequeno de intenções.

**H2 — As intenções podem ser inferidas a partir de padrões textuais curtos.** `Ticket Subject` funciona como um resumo do problema e apresenta categorias semanticamente informativas, o que indica que uma futura solução de NLP pode combinar assunto e descrição para classificar a intenção.

**H3 — A urgência percebida pelo usuário pode estar relacionada à natureza da intenção.** A existência de quatro níveis de prioridade (`Low`, `Medium`, `High`, `Critical`) permite investigar futuramente se intenções como falhas técnicas, perda de dados ou acesso à conta aparecem proporcionalmente mais associadas a prioridades elevadas.

**H4 — O canal de contato pode refletir diferentes comportamentos de busca de suporte.** Como o dataset possui quatro canais (`Email`, `Phone`, `Social media`, `Chat`), é possível investigar se determinados tipos de intenção são mais frequentes em canais específicos.

> Estas são hipóteses exploratórias, não conclusões causais. A etapa seguinte deveria testar as relações com análises bivariadas/multivariadas.

## Conclusão da EDA inicial

O dataset apresenta estrutura adequada para o domínio de atendimento ao cliente e para uma futura tarefa de classificação de intenção. Há 8.469 tickets, cinco tipos principais de ticket, quatro prioridades e quatro canais. A qualidade deve ser tratada com atenção principalmente nos campos dependentes do encerramento do ticket e nos textos com placeholders.

A próxima etapa natural é transformar `Ticket Subject` + `Ticket Description` em entrada de um modelo de NLP e utilizar `Ticket Type` como variável-alvo inicial, mantendo a API `/predict` como camada de exposição do modelo.